In [15]:
from git import Repo
from langchain.text_splitter import Language
from langchain.document_loaders.generic import GenericLoader
from langchain.document_loaders.parsers import LanguageParser
from langchain.text_splitter import RecursiveCharacterTextSplitter
#from langchain.embeddings import OpenAIEmbeddings
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
#from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI
from langchain.memory import ConversationSummaryMemory
from langchain.chains import ConversationalRetrievalChain
import os

clone the github repo

In [16]:
!mkdir -p test_repo

A subdirectory or file -p already exists.
Error occurred while processing: -p.


In [17]:
repo_path = 'test_repo'
repo= Repo.clone_from("https://github.com/swarajjoshi10-ship-it/stateful-agentic",to_path=repo_path)

loading the data from the files

In [18]:
%pwd

'c:\\Realtime-Source-Code-Analyzer\\research'

In [19]:
loader= GenericLoader.from_filesystem(repo_path,
                        glob="**/*.*",
                        suffixes=[".py"],
                        parser=LanguageParser(language=Language.PYTHON,parser_threshold=500))

In [20]:
documents= loader.load()

In [21]:
len(documents)

20

split the documents into chunks

In [22]:
documents_splitter = RecursiveCharacterTextSplitter.from_language(language=Language.PYTHON, 
                                                                  chunk_size=500,
                                                                  chunk_overlap=20)

In [23]:
texts= documents_splitter.split_documents(documents)

In [24]:
len(texts)

57

download the OpenAI Embeddings

In [25]:
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [26]:
#os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [27]:
#embeddings = OpenAIEmbeddings(disallowed_special=())
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 36327.59it/s]


setting up chroma as vector database

In [28]:
#vectordb= Chroma.from_documents(documents=texts, embedding=embeddings,persist_directory='./data')
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory='./data'
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


creating an OpenAI model wrapper

In [46]:
#llm=ChatOpenAI()
llm = ChatOpenAI(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=OPENAI_API_KEY,
    model_name="openrouter/free"
)

In [47]:
memory=ConversationSummaryMemory(llm=llm,
                                  memory_key="chat_history",
                                  return_messages=True)

In [48]:
qa = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectordb.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 8}
    ),
    memory=memory
)

Question and answer

In [32]:
question = "what is the chatbot_with_tools_build_graph function"

In [49]:
result=qa(question)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [50]:
print(result['answer'])


The `chatbot_with_tools_build_graph` function is a method (likely of a `GraphBuilder` class) that constructs a LangGraph state machine for a chatbot capable of using external tools (e.g., web search).  

**Key actions performed by the function:**  
- Retrieves a list of tools via `get_tools()` (e.g., `TavilySearchResults`).  
- Creates a `ToolNode` from those tools.  
- Initializes a `ChatbotWithToolNode` (a custom node that knows how to invoke the LLM with tool bindings).  
- Adds both the chatbot node and the tool node to the graph builder.  
- Sets up edges (including conditional edges) between the nodes so the chatbot can decide when to call a tool and how to resume after a tool response.  
- Designates the chatbot node as the entry point of the graph.  

After this method runs, the graph is typically compiled (e.g., in `setup_graph`) so it can be invoked with an initial state. It is used when the selected use case is "ChatBot with Web Search".
